# `ta_tools` data readers: a hands-on check

Runs every way `ta_tools` gets bars and checks each result against the output contract they
all share (`_normalise` in `src/tools/ta_tools/data.py`): `open/high/low/close`, then
`volume` when the source has one, then any other columns; float64 prices; a `date` index,
oldest first, with no repeated timestamp and no price gap after the first value.

Each check prints ✓ or ✗ with a short detail, and the **Summary** at the end totals them.
Nothing here stops on a failed check, so one run shows everything.

| Section | Tests | Needs |
|---|---|---|
| 1 | `make_bars`, the offline simulator | nothing |
| 2 | `load_bars`, daily | Yahoo |
| 3 | `load_bars`, intraday (1m, 5m, 15m, 1h) | Yahoo |
| 4 | Yahoo's history limits: refused requests, and requests just inside each limit | Yahoo for the second half |
| 5 | `read_bars` on a TradingView *Export chart data* file | section 2's bars |
| 6 | `read_bars` on another layout (Yahoo-style columns) | section 2's bars |
| 7 | `read_bars` on **your own** TradingView export or connector CSV | a file; skipped if unset |
| 8 | What the contract rejects | nothing |

Sections 5 and 6 write their CSV files to a temporary folder, built from the Yahoo bars, so a
round trip must give back exactly the bars that went in.

In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with `pip install -e ".[ta]"`
# (Python 3.12 or newer). On Colab the install runs here, and the repo also goes on sys.path
# because an editable install is only picked up when the interpreter starts.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(f'No repo at {REPO}. Update REPO above.')
    !pip install -q -e "{REPO}[ta]"
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

print(f'Running in Colab: {IN_COLAB}')

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from src.tools import ta_tools
from src.tools.ta_tools.data import YAHOO_LOOKBACK_DAYS

TICKER = 'AAPL'
EXCHANGE_TZ = 'America/New_York'        # AAPL's exchange time
START = '2024-01-01'
PRICES = ['open', 'high', 'low', 'close']

TODAY = pd.Timestamp.today().normalize()
WORKDIR = Path(tempfile.mkdtemp(prefix='ta_tools_read_'))
print('CSV files go to', WORKDIR)

results = []


def check(label, ok, detail=''):
    """Record one check and print it; the Summary section totals them."""
    ok = bool(ok)
    results.append({'check': label, 'passed': ok, 'detail': detail})
    print(f"{'✓' if ok else '✗'} {label}" + (f'  ({detail})' if detail else ''))


def describe(bars, label):
    span = f'{bars.index[0]} to {bars.index[-1]}' if len(bars) else 'empty'
    print(f'{label}: {len(bars)} bars, {span}, tz={bars.index.tz}, columns={list(bars.columns)}')


def check_contract(bars, label):
    """The output contract every loader shares."""
    core = PRICES + (['volume'] if 'volume' in bars.columns else [])
    check(f'{label}: columns start open/high/low/close[/volume]', list(bars.columns[:len(core)]) == core,
          str(list(bars.columns)))
    check(f'{label}: prices and volume are float64', (bars[core].dtypes == 'float64').all())
    check(f"{label}: DatetimeIndex named 'date'",
          isinstance(bars.index, pd.DatetimeIndex) and bars.index.name == 'date')
    check(f'{label}: oldest first, no repeated timestamp',
          bars.index.is_monotonic_increasing and not bars.index.has_duplicates)
    check(f'{label}: no price gap after the first value',
          all(bars.loc[bars[c].first_valid_index():, c].notna().all() for c in PRICES))


def check_candles(bars, label):
    """Data sanity rather than contract: a ✗ here is the source's data, not ta_tools."""
    body_top, body_bottom = bars[['open', 'close']].max(axis=1), bars[['open', 'close']].min(axis=1)
    bad = int(((bars['high'] < body_top) | (bars['low'] > body_bottom)).sum())
    check(f'{label}: low <= open, close <= high on every bar', bad == 0, f'{bad} bars break it')


def refused(label, call):
    """Check that `call` raises ValueError, and show the message a user would see."""
    try:
        call()
    except ValueError as exc:
        check(f'refused: {label}', True, str(exc))
    else:
        check(f'refused: {label}', False, 'no error raised')


def days_ago(days):
    return (TODAY - pd.Timedelta(days=days)).date()

## 1. `make_bars`: the offline simulator

The reference shape: seeded, daily, with volume. Every other source should look like this.

In [ ]:
sim = ta_tools.make_bars(n=250, seed=0)
describe(sim, 'make_bars')
check_contract(sim, 'make_bars')
check_candles(sim, 'make_bars')
sim.head()

## 2. `load_bars`, daily

Daily bars from Yahoo Finance, **as traded** (`auto_adjust=False`, not dividend-adjusted). The
index should be plain dates: no time of day and no timezone.

In [ ]:
daily = ta_tools.load_bars(TICKER, START)
describe(daily, 'daily')
check_contract(daily, 'daily')
check_candles(daily, 'daily')
check('daily: index is dates, no timezone',
      daily.index.tz is None and (daily.index == daily.index.normalize()).all())
check('daily: every date is a weekday', (daily.index.dayofweek < 5).all())
check('daily: same index resolution as make_bars', daily.index.dtype == sim.index.dtype,
      str(daily.index.dtype))
daily.tail()

## 3. `load_bars`, intraday

`interval=` is passed to yfinance. Intraday bars keep a **timezone-aware** index in exchange time,
so a regular-session 5-minute bar starts at 09:30 New York time. Each request below sits well
inside Yahoo's history limit for its interval.

In [ ]:
INTRADAY = {'1m': (3, '1min'), '5m': (10, '5min'), '15m': (30, '15min'), '1h': (120, '1h')}

intraday = {}
for interval, (days, step) in INTRADAY.items():
    bars = ta_tools.load_bars(TICKER, days_ago(days), interval=interval)
    intraday[interval] = bars
    describe(bars, interval)
    check_contract(bars, interval)
    check(f'{interval}: index in exchange time', str(bars.index.tz) == EXCHANGE_TZ, str(bars.index.tz))
    gap = bars.index.to_series().diff().min()
    check(f'{interval}: bars are {step} apart', gap == pd.Timedelta(step), f'smallest gap {gap}')
    check(f'{interval}: first bar of each day at 09:30 or later',
          (bars.index.to_series().groupby(bars.index.date).min().dt.strftime('%H:%M') >= '09:30').all())
    print()

In [ ]:
# Bars per session. A full regular session is 390 one-minute bars, 78 five-minute bars,
# 26 fifteen-minute bars and 7 hourly bars (the last one a half hour); today's may be partial.
pd.DataFrame({interval: bars.groupby(bars.index.date).size()
              for interval, bars in intraday.items()}).tail(8)

## 4. Yahoo's history limits

Yahoo keeps intraday history for a limited time, counted back from today, and serves 1-minute
bars at most 7 days per request:

In [ ]:
pd.Series(YAHOO_LOOKBACK_DAYS, name='days of history').to_frame()

Asking for more makes yfinance return an **empty** frame, so `load_bars` refuses such requests
itself, before calling Yahoo. These run without the network:

In [ ]:
refused('5m from 90 days ago', lambda: ta_tools.load_bars(TICKER, days_ago(90), interval='5m'))
refused('1m from 45 days ago', lambda: ta_tools.load_bars(TICKER, days_ago(45), interval='1m'))
refused('1m over a 10-day span', lambda: ta_tools.load_bars(TICKER, days_ago(12), days_ago(2), interval='1m'))
refused('1h from 800 days ago', lambda: ta_tools.load_bars(TICKER, days_ago(800), interval='1h'))
refused("unknown interval '3m'", lambda: ta_tools.load_bars(TICKER, days_ago(2), interval='3m'))

The limits come from Yahoo's own error messages and **had not been checked live** when this was
written. Each request below starts just inside one limit; Yahoo should serve it in full, with the
first bar within a few days of the start (weekends and holidays aside). A ✗ here means the table
above needs changing.

In [ ]:
for interval, days in YAHOO_LOOKBACK_DAYS.items():
    start = days_ago(days - 1)
    end = days_ago(days - 7) if interval == '1m' else None      # 1m: 6 days, inside the 7-day cap
    try:
        bars = ta_tools.load_bars(TICKER, start, end, interval=interval)
    except ValueError as exc:
        check(f'{interval} from {days - 1} days ago is served', False, str(exc))
        continue
    lag = (bars.index[0].tz_localize(None).normalize() - pd.Timestamp(start)).days
    check(f'{interval} from {days - 1} days ago is served', lag <= 4,
          f'{len(bars)} bars, first {lag} days after the start')

## 5. `read_bars` on a TradingView export

TradingView's *Export chart data* writes one row per bar: `time` in Unix seconds, lower-case
prices, one column per plotted indicator series, and `Volume`. Daily bars are stamped at the
**09:30 New York open**. The file below is written in that layout from section 2's bars, with a
20-bar SMA as the plotted series (blank during its warm-up, as in a real export).

In [ ]:
opens = (daily.index + pd.Timedelta(hours=9, minutes=30)).tz_localize(EXCHANGE_TZ)
unix_seconds = (opens - pd.Timestamp('1970-01-01', tz='UTC')) // pd.Timedelta('1s')
sma20 = ta_tools.sma(daily['close'], 20)

export = pd.DataFrame({'time': unix_seconds, **{c: daily[c].to_numpy() for c in PRICES},
                       'SMA 20': sma20.to_numpy(), 'Volume': daily['volume'].to_numpy()})
tv_path = WORKDIR / 'tradingview_export.csv'
export.to_csv(tv_path, index=False)
print(''.join(tv_path.read_text().splitlines(keepends=True)[:3]))

In [ ]:
tv = ta_tools.read_bars(tv_path, daily=True, tz=EXCHANGE_TZ)
describe(tv, 'TradingView export')
check_contract(tv, 'TradingView export')
check('export: same dates as load_bars', tv.index.equals(daily.index))
check('export: prices and volume round-trip exactly',
      tv[PRICES + ['volume']].equals(daily[PRICES + ['volume']]))
check('export: indicator column kept after the prices', list(tv.columns) == PRICES + ['volume', 'SMA 20'])
check('export: indicator warm-up still blank (19 bars)', tv['SMA 20'].isna().sum() == 19)
check('export: indicator values intact', np.allclose(tv['SMA 20'], sma20, equal_nan=True))
tv.head()

The same file's times three ways. Without `tz` they are UTC, so the 09:30 open reads **13:30 or
14:30** depending on daylight saving. With `tz` they are New York time. With `daily=True` only
the date is kept, taken in `tz`.

In [ ]:
as_utc = ta_tools.read_bars(tv_path)
as_local = ta_tools.read_bars(tv_path, tz=EXCHANGE_TZ)
check('no tz: times are UTC', str(as_utc.index.tz) == 'UTC')
check('no tz: open at 13:30 or 14:30 UTC (daylight saving)',
      set(as_utc.index.strftime('%H:%M')) <= {'13:30', '14:30'},
      str(as_utc.index.strftime('%H:%M').value_counts().to_dict()))
check(f'tz={EXCHANGE_TZ}: every bar at 09:30', (as_local.index.strftime('%H:%M') == '09:30').all())
pd.DataFrame({'UTC': as_utc.index[:3], 'New York': as_local.index[:3], 'daily=True': tv.index[:3]})

The export dialog may offer ISO 8601 times instead of Unix seconds. `read_bars` should give the
same frame either way.

In [ ]:
iso_path = WORKDIR / 'tradingview_export_iso.csv'
export.assign(time=[t.isoformat() for t in opens]).to_csv(iso_path, index=False)
print(iso_path.read_text().splitlines()[1])

tv_iso = ta_tools.read_bars(iso_path, daily=True, tz=EXCHANGE_TZ)
check('ISO times give the same frame as Unix seconds', tv_iso.equals(tv))

## 6. `read_bars` on another layout

Any CSV works once its columns are mapped. Here, a Yahoo-style file (`Date`, `Open`, ..., `Volume`):
`columns=` maps `Date` to `time`, and the price names are matched ignoring case.

In [ ]:
yahoo_path = WORKDIR / 'yahoo_style.csv'
daily.rename(columns=str.title).to_csv(yahoo_path, index_label='Date')
print(''.join(yahoo_path.read_text().splitlines(keepends=True)[:3]))

other = ta_tools.read_bars(yahoo_path, columns={'Date': 'time'}, daily=True)
check_contract(other, 'Yahoo-style CSV')
check('Yahoo-style CSV: round-trips to the load_bars frame', other.equals(daily))

## 7. Your own TradingView export

Set `MY_EXPORT` to a real file, either from TradingView (chart menu → *Export chart data…*) or one
Claude wrote from the TradingView connector, and re-run this cell. It prints the file's first
lines, so you can see which time format the export dialog used, then reads the file and, for a
daily file of `TICKER`, compares its closes with Yahoo's on the dates they share.

In [ ]:
MY_EXPORT = None                  # e.g. '~/Downloads/NASDAQ_AAPL, 1D.csv'
MY_DAILY = True                   # False for an intraday chart
MY_TZ = EXCHANGE_TZ               # the symbol's exchange timezone

if MY_EXPORT is None:
    print('Skipped: set MY_EXPORT to read your own file.')
else:
    path = Path(MY_EXPORT).expanduser()
    lines = path.read_text().splitlines()
    print('Header:   ', lines[0])
    print('First row:', lines[1])
    mine = ta_tools.read_bars(path, daily=MY_DAILY, tz=MY_TZ)
    describe(mine, 'your export')
    check_contract(mine, 'your export')
    print('Indicator columns:', [c for c in mine.columns if c not in PRICES + ['volume']])
    shared = mine.index.intersection(daily.index) if MY_DAILY else []
    if len(shared):
        diff = (mine.loc[shared, 'close'] - daily.loc[shared, 'close']).abs()
        check(f'your export: closes match Yahoo on {len(shared)} shared dates', diff.max() < 0.01,
              f'largest difference {diff.max():.4f}')
    else:
        print(f'No dates shared with the {TICKER} Yahoo bars, so no comparison.')
    display(mine.tail())

## 8. What the contract rejects

Each file below breaks one rule, and `read_bars` should refuse it with a message that says
which. The last two cases are allowed, and should read cleanly.

In [ ]:
HEADER = 'time,open,high,low,close,Plot'
ROWS = ['1704205800,1,2,0.5,1.5,7', '1704292200,1.5,2,1,1.8,8', '1704378600,1.8,2.2,1.6,2,9']


def csv_file(name, lines):
    path = WORKDIR / f'{name}.csv'
    path.write_text('\n'.join(lines) + '\n')
    return path


BAD = {
    'rows newest first': ([HEADER] + ROWS[::-1], {}),
    'a repeated timestamp': ([HEADER] + ROWS[:2] + ROWS[1:2], {}),
    'a close missing mid-series': ([HEADER, ROWS[0], '1704292200,1.5,2,1,,8', ROWS[2]], {}),
    'no low column': (['time,open,high,close'] + [','.join(r.split(',')[:3] + r.split(',')[4:5]) for r in ROWS], {}),
    'no time column': (['date,open,high,low,close'] + [r.rsplit(',', 1)[0] for r in ROWS], {}),
    'two 5-minute bars squeezed to one date': ([HEADER, ROWS[0], '1704206100,1.5,2,1,1.8,8'], {'daily': True}),
}
for i, (label, (lines, kwargs)) in enumerate(BAD.items()):
    refused(label, lambda: ta_tools.read_bars(csv_file(f'bad_{i}', lines), **kwargs))

ok = ta_tools.read_bars(csv_file('leading_gap', [HEADER, '1704205800,1,2,0.5,,7'] + ROWS[1:]))
check('allowed: a price missing before its first value', ok['close'].isna().sum() == 1)
ok = ta_tools.read_bars(csv_file('plot_gap', [HEADER, '1704205800,1,2,0.5,1.5,', ROWS[1], ROWS[2][:-1]]))
check('allowed: gaps in an indicator column', ok['Plot'].isna().sum() == 2)

## Summary

In [ ]:
summary = pd.DataFrame(results)
failed = summary[~summary['passed']]
print(f"{int(summary['passed'].sum())} of {len(summary)} checks passed.")
failed if len(failed) else print('Nothing failed.')